# 07 · Weather: ERA5 and ERA5-Land

Hourly weather for every site on the days of its competitions (and the day before each), from the Copernicus Climate Data Store. Adds to `database/weather/`:

| Table | One row per | Content |
|---|---|---|
| `era5_single` | chip (ERA5 point, 0.25°) and UTC hour | boundary-layer height, cloud base, CAPE, CIN, K index, sensible and latent heat flux, sunshine, cloud cover (low/mid/high), temperature, dew point, wind at 10 and 100 m, gusts, pressure, rain |
| `era5_levels` | chip and UTC hour | wind, temperature, humidity and height at 925, 850, 700, 600 and 500 hPa |
| `era5_land` | ERA5-Land point (0.1°) and UTC hour | sensible heat flux, skin temperature, soil moisture, albedo, snow cover, sunshine |

**Before running:** a free CDS account with the licences of the three datasets accepted, and its API key saved as the Colab secret `CDSAPI_KEY` with notebook access turned on (key icon on the left).

Sites flown in the same month are grouped into regional requests. The CDS queues requests, so this can take several hours; downloaded files are kept, so **if Colab disconnects, Run All again** and it only asks for what is missing.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
PARALLEL = 4         # requests queued at the CDS at a time (it works through one or a few per user at once)
BOX = 25             # largest regional box in degrees: fewer, larger requests finish sooner
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, load the CDS key, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

drive.mount("/content/drive", force_remount=True)
auth.authenticate_user()
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "site_chips.parquet")), "Run notebook 06 first: database/site_chips.parquet is missing"
subprocess.run(["pip", "install", "-q", "cdsapi", "netCDF4"], check=True)

from google.colab import userdata
os.environ["CDSAPI_KEY"] = userdata.get("CDSAPI_KEY")   # stays in this session; never printed
print("Database in", DB, "| CDS key loaded")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script (otherwise it would
    keep running in the background and, here, keep sending requests to the CDS)."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")

## 3. The requests

How many regional requests there are (and how many other box sizes would give), and the first few. Nothing is downloaded yet. Requests go in order: single levels first (the most useful), then pressure levels, then ERA5-Land.

In [ ]:
run(f"{REPO}/scripts/era5.py", DB, "--plan-only", "--box", str(BOX))

## 4. Download and convert

Asks the CDS for the missing files, `PARALLEL` at a time, then builds the three tables. Prints a line per finished request. Stopping this cell also stops the downloads. A request that fails is written to `weather/failed.txt` and tried again on the next run.

In [ ]:
run(f"{REPO}/scripts/era5.py", DB, "--parallel", str(PARALLEL), "--box", str(BOX))

## 5. First check: ERA5 boundary layer vs real thermal tops

In [ ]:
import duckdb
db = duckdb.connect()
W = os.path.join(DB, "weather")
for name in ["single", "levels", "land"]:
    db.execute(f"CREATE VIEW era5_{name} AS SELECT * FROM read_parquet('{W}/era5_{name}.parquet')")
db.execute(f"CREATE VIEW climbs AS SELECT * FROM read_parquet('{DB}/climbs.parquet')")
db.execute(f"CREATE VIEW tasks AS SELECT * FROM read_parquet('{DB}/tasks.parquet')")

# First check against the pilots: ERA5 boundary-layer height vs the tops of real thermals, same chip and hour
import sys; sys.path.insert(0, REPO)
from pta import grid
c = db.sql("""
    SELECT c.task_key, c.lat, c.lon, c.alt_top, c.t_start, k.date
    FROM climbs c JOIN tasks k USING (task_key)
    WHERE c.is_thermal AND k.date IS NOT NULL
    USING SAMPLE 200000
""").df()
c["chip_id"] = grid.chip_id(c["lat"].to_numpy(), c["lon"].to_numpy())
c["time"] = (pd.to_datetime(c["date"]) + pd.to_timedelta(c["t_start"], unit="s")).dt.round("h")
w = db.sql("SELECT chip_id, time, blh_m FROM era5_single").df()
lv = db.sql("SELECT chip_id, time, z_850 FROM era5_levels").df()
m = c.merge(w, on=["chip_id", "time"]).merge(lv, on=["chip_id", "time"])
# thermal top above the ERA5 ground (z_850 is not ground; use the chip's mean elevation from cells)
cells = pd.read_parquet(f"{DB}/cells.parquet", columns=["chip_id", "elev_mean"]).groupby("chip_id").elev_mean.mean()
m["ground"] = m["chip_id"].map(cells)
m["top_agl"] = m["alt_top"] - m["ground"]
print(f"{len(m):,} thermals matched to ERA5")
print(m[["top_agl", "blh_m"]].describe().round(0))
print("correlation:", round(m["top_agl"].corr(m["blh_m"]), 2))